In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, split, explode, trim

#bronze table filepaths
bronze_table_customers = "dev.trailsport.bronze_customers"
bronze_table_products = "dev.trailsport.bronze_products"
bronze_table_sales = "dev.trailsport.bronze_sales"

#silver table filepaths
silver_table_customers = "dev.trailsport.silver_customers"
silver_table_products = "dev.trailsport.silver_products"
silver_table_sales = "dev.trailsport.silver_sales"
silver_table_sales_items = "dev.trailsport.silver_sales_items"

In [ ]:
# load sales data early so it can be cleaned alongside other tables
df_silver_sales = spark.read.table(bronze_table_sales)

# 1. Customers, load data and write to silver

In [0]:
# read df_bronze_customers
df_silver_customers = spark.read.table(bronze_table_customers)


In [0]:
#show all customers
display(df_silver_customers.limit(10))

CustomerID,CustomerName,BirthDate,Gender,MaritalStatus,Children,Education,Occupation,PostalCode,CityCat,Income,OwnsHouse,OwnsCar,ReceivedDiscountEmail,CustomerSince,HasLoyaltyCard,ingest_time
45934,N.N. Landkroon,1995-09-03,Male,Married,0,MBO,Agricultural,7632VG,Metropolis,74500.0,True,True,True,2017-04-13,False,2026-04-02T14:15:08.343Z
83434,N.N. Orie,null,Male,Married,null,MBO,Machine operator,9129RK,null,null,null,False,False,2019-10-27,True,2026-04-02T14:15:08.343Z
77898,A. Dräger,1983-06-04,Female,Divorced,2,MBO,Machine operator,null,null,27000.0,False,True,False,2018-08-09,False,2026-04-02T14:15:08.343Z
90601,"C. Meij, van der",1982-05-21,Male,Single,0,MBO,Technician,3609PB,City,31500.0,False,False,False,2016-11-27,False,2026-04-02T14:15:08.343Z
79320,L. Schattefor,1986-08-13,Female,Married,1,MBO,Agricultural,null,null,40500.0,False,False,False,2016-11-27,False,2026-04-02T14:15:08.343Z
68790,A. Wigmans,1975-05-26,Male,Divorced,1,MBO,Technician,null,null,61000.0,True,True,True,2020-06-03,False,2026-04-02T14:15:08.343Z
88255,"B.M.S. Rooijen, van",null,Male,Married,0,Master UNI,Agricultural,3332JT,Town,92500.0,True,True,False,2018-02-06,True,2026-04-02T14:15:08.343Z
77260,null,null,Male,null,null,MBO,Manager,null,null,53000.0,null,True,True,2017-10-01,False,2026-04-02T14:15:08.343Z
29697,"C.P. Logt, van der",2000-11-15,Male,Single,0,Master UNI,Machine operator,null,null,38000.0,False,True,True,2022-08-19,False,2026-04-02T14:15:08.343Z
72309,"V.B. Vliet, van",1979-03-30,Male,Single,0,Master HBO,Technician,3521JC,Town,34500.0,False,False,False,2018-09-13,False,2026-04-02T14:15:08.343Z


### 1.1 Clean up text only

In [0]:
# true false columns: "Education","Occupation","CityCat", "CustomerName"
textonly_cols = ["CustomerName","Education","Occupation","CityCat"]

# Check for digit in cell then unknown else value
for col_name in textonly_cols:
    safetext = F.when(
        (F.col(col_name).isNull()) | (F.col(col_name).rlike(r"\d")), 
        None # instead of None # instead of "unknown" 
    ).otherwise(F.col(col_name))
    df_silver_customers = df_silver_customers.withColumn(col_name, safetext)

### 1.2 Cleanup Frue False

In [0]:
# true false columns: ownshouse, ownscar, receiveddiscountemail, hascustomercard
boolean_cols = ["OwnsHouse", "OwnsCar", "ReceivedDiscountEmail", "HasLoyaltyCard"]
#loops through all the columns
for col_name in boolean_cols:
    safe_bool = F.expr(f"try_cast({col_name} as boolean)")
    df_silver_customers = df_silver_customers.withColumn(col_name, safe_bool)


### 1.3 Cleanup children

In [0]:
#first cast age as pyhton int
safe_children = F.expr("try_cast(Children as int)")
#verything thes is above 999 will be none
df_silver_customers = df_silver_customers.withColumn(
    "Children",
    F.when(
        (safe_children > 999), 
        F.lit(None)
    ).otherwise(safe_children)
)

### 1.4 Cleanup income

In [0]:
safe_income = F.expr("try_cast(Income as double)")
# null out incomes above 500,000 as unrealistic for this customer dataset
df_silver_customers = df_silver_customers.withColumn(
    "Income",
    F.when(
        (safe_income > 500000), 
        F.lit(None)
    ).otherwise(safe_income)
)

### 1.5 Cleanup MaritalStatus

In [0]:
# Make everything that has a digit .rlike(r"\d") None # instead of "unknown"
df_silver_customers = df_silver_customers.withColumn(
    "MaritalStatus",
    F.when(
        (F.col("MaritalStatus").rlike(r"\d"))| (F.col("MaritalStatus").isNull()) | (F.col("MaritalStatus") == "FB") ,
        None # instead of "unknown"
    ).otherwise(F.col("MaritalStatus"))
)

### 1.6 Cleanup Birthdate

In [0]:
date_cols = ["BirthDate", "CustomerSince"]
#set all dates to None # instead of "unknown"  when not like "r"^\d{4}-\d{2}-\d{2}$""

date_transformations = {
    col_name: F.when(
        (~F.col(col_name).rlike(r"^\d{4}-\d{2}-\d{2}$")),
        None # instead of "unknown"
    ).otherwise(F.col(col_name))
    for col_name in date_cols
}

df_silver_customers = df_silver_customers.withColumns(date_transformations)


### 1.7 Clean SaleDate and SalePrice

In [ ]:
# Null out SaleDate values that don't match YYYY-MM-DD format
df_silver_sales = df_silver_sales.withColumn(
    "SaleDate",
    F.when(
        (~F.col("SaleDate").rlike(r"^\d{4}-\d{2}-\d{2}$")) | F.col("SaleDate").isNull(),
        None
    ).otherwise(F.col("SaleDate"))
)

# Cast SalePrice to double, null out negatives
safe_price = F.expr("try_cast(SalePrice as double)")
df_silver_sales = df_silver_sales.withColumn(
    "SalePrice",
    F.when(safe_price < 0, None).otherwise(safe_price)
)

### 1.8 Gender cleanup

In [0]:
#Visualy check records where gender values that are not Male or Female
df_silver_customers_genderless = df_silver_customers.filter(
    ~F.col("Gender").isin("Male","Female"))
#display(df_silver_customers_genderless)

In [0]:
#remove all records where gender values that are not Male or Female
df_silver_customers = df_silver_customers\
    .filter((F.col("Gender").isin("Male","Female"))|(F.col("Gender").isNull()))


### 1.9 Load data into silver

In [0]:
#insert in silver table
df_silver_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table_customers)

#2. Products, load data and write to silver


In [0]:
# read df_bronze_products
df_silver_products = spark.read.table(bronze_table_products)


### 2.1 Clean corrupted category and type values

In [ ]:
# Null out ProductCategory and ProductType values that are corrupted hash-like numbers
hash_pattern = r"^\d{10,}$"
for col_name in ["ProductCategory", "ProductType"]:
    df_silver_products = df_silver_products.withColumn(
        col_name,
        F.when(F.col(col_name).rlike(hash_pattern), None).otherwise(F.col(col_name))
    )

### 2.2 Load data into silver

In [0]:
#insert in silver table
df_silver_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table_products)

#3. Sales, load data and write to silver


### 3.1 Split sales and sales items

In [0]:
df_sales_items_base = df_silver_sales.select("SaleID","ProductIDs")
df_sales_items_array = df_sales_items_base.withColumn("products_array", split(col("ProductIDs"),", "))
df_sales_items_exploded = df_sales_items_array.withColumn("ProductID", explode(col("products_array")))

### 3.2 Create sales table

In [0]:
#create silver sales table without products and productids
df_silver_sales.select("SaleID","CustomerID","SaleDate","SalePrice").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table_sales)

#create new sales items table SaleID and SingleItem in new table
df_sales_items_exploded.select("SaleID","ProductID").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table_sales_items)